## Setup Redis connection

In [1]:
import os

REDIS_HOST = os.getenv("REDIS_HOST", "localhost")
REDIS_PORT = os.getenv("REDIS_PORT", "6379")
REDIS_PASSWORD = os.getenv("REDIS_PASSWORD", "")

REDIS_URL = f"redis://:{REDIS_PASSWORD}@{REDIS_HOST}:{REDIS_PORT}"
os.environ["REDIS_URL"] = REDIS_URL

print("✅ Redis connection variables set.")

✅ Redis connection variables set.


In [2]:
# import warnings
# from redis import Redis
# warnings.filterwarnings('ignore')

# redis_connection = Redis.from_url(REDIS_URL)
# redis_connection.ping()

## Set up the text splitter and loader

In [3]:
from langchain_community.document_loaders.base_o365 import CHUNK_SIZE
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader("data/nke-10k-2023.pdf")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 2500,
    chunk_overlap= 50
)

# Confirms the text splitter was created successfully
print("Text Splitter:", text_splitter)

# Confirms the loader was created successfully
print("Loader:", loader)

/var/folders/7s/0_nmrjsn083b91_5kddmb6dw0000gn/T/ipykernel_7793/1383856782.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.base_o365 import CHUNK_SIZE


Text Splitter: <langchain_text_splitters.character.RecursiveCharacterTextSplitter object at 0x10ffc0d70>
Loader: <langchain_community.document_loaders.pdf.PyPDFLoader object at 0x10ffc0ad0>


In [4]:
# Extract, load, and split the document into chunks
chunks = loader.load_and_split(text_splitter)

print("Done preprocessing. Created", len(chunks), "chunks of the original PDF:")

Done preprocessing. Created 211 chunks of the original PDF:


## Text embedding generation with RedisVL



In [5]:
import warnings
import pandas as pd
from redisvl.utils.vectorize import HFTextVectorizer, BaseVectorizer
from redisvl.extensions.cache.embeddings import EmbeddingsCache

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Create the new vectorizer below 
hf = HFTextVectorizer(
    model="sentence-transformers/all-MiniLM-L6-v2",
    cache=EmbeddingsCache(
        name="embedcache",
        ttl=600,
        redis_url=REDIS_URL,
    )
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
# Embed each chunk content
embeddings = hf.embed_many([chunk.page_content for chunk in chunks])

# Check to make sure we've created enough embeddings, 1 per document chunk
len(embeddings) == len(chunks)

True

## Define the index schema

#### Create an index schema

Create a new variable (for a Python dictionary) called schema that defines the structure of the index for storing your document chunks in Redis. Your schema should include:

Create a schema named "doc_redisvl" (you can plug in the variable index_name we provided) with:

* A prefix of "chunk"
* A storage type of "hash"

In [7]:
from redisvl.index import SearchIndex

index_name = "doc_redisvl"

schema = {
  "index": {
    "name": index_name,
    "prefix": "nke_chunk",
    "storage_type": "hash"
  },
  "fields": [
    {
        "name": "chunk_id",
        "type": "tag",
        "attrs": {
            "sortable": True
        }
    },
    {
        "name": "content",
        "type": "text"
    },
    {
        "name": "text_embedding",
        "type": "vector",
        "attrs": {
            "dims": 384,
            "distance_metric": "cosine",
            "algorithm": "hnsw",
            "datatype": "float32"
        }
    }
  ]
}

print("Schema created successfully:", schema)

Schema created successfully: {'index': {'name': 'doc_redisvl', 'prefix': 'nke_chunk', 'storage_type': 'hash'}, 'fields': [{'name': 'chunk_id', 'type': 'tag', 'attrs': {'sortable': True}}, {'name': 'content', 'type': 'text'}, {'name': 'text_embedding', 'type': 'vector', 'attrs': {'dims': 384, 'distance_metric': 'cosine', 'algorithm': 'hnsw', 'datatype': 'float32'}}]}


### Index creation

With the schema created, we can now create the index and load it into Redis. 

In [8]:
# create an index from schema and the client
index = SearchIndex.from_dict(schema, redis_url=REDIS_URL)
index.create(overwrite=True, drop=True)

print("Index created successfully:", index)

Index created successfully: SearchIndex(name='doc_redisvl', prefixes=['nke_chunk'], storage_type='hash')


We can additionally confirm the index exists with the following RedisVL CLI commands.

In [9]:
# List all indices. 
!rvl index listall

# Get info about the index
!rvl index info -i doc_redisvl

Indices:
1. llmcache
2. movies
3. doc_redisvl
4. chat_session_Tom


Index Information:
╭───────────────┬───────────────┬───────────────┬───────────────┬───────────────╮
│ Index Name    │ Storage Type  │ Prefixes      │ Index Options │ Indexing      │
├───────────────┼───────────────┼───────────────┼───────────────┼───────────────┤
| doc_redisvl   | HASH          | ['nke_chunk'] | []            | 0             |
╰───────────────┴───────────────┴───────────────┴───────────────┴───────────────╯
Index Fields:
╭─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────╮
│ Name            │ Attribute       │ Type            │ Field Option    │ Option Value    │ Field Option    │ Option Value    │ Field Option    │ Option Value    │ Field Option    │ Option Value    │ Field Option    │

## Create and load the dataset

It’s time to store the data in Redis so we can start searching it. We will need to manually assemble a list of Python dictionaries (one per chunk) so the structure matches the schema.

Each entry should have the following properties:

* chunk_id: a unique ID for the Redis key
* content: the chunk’s text content
* text_embedding: the vector embedding, converted to a byte buffer using array_to_buffer()

In [10]:
from redisvl.redis.utils import array_to_buffer

data = [
    {
        'chunk_id': i,
        'content': chunk.page_content,
        # For HASH -- we must convert embeddings to bytes
        'text_embedding': array_to_buffer(embeddings[i], dtype='float32')
    } for i, chunk in enumerate(chunks)
]

index.load(data, id_field="chunk_id")

['nke_chunk:0',
 'nke_chunk:1',
 'nke_chunk:2',
 'nke_chunk:3',
 'nke_chunk:4',
 'nke_chunk:5',
 'nke_chunk:6',
 'nke_chunk:7',
 'nke_chunk:8',
 'nke_chunk:9',
 'nke_chunk:10',
 'nke_chunk:11',
 'nke_chunk:12',
 'nke_chunk:13',
 'nke_chunk:14',
 'nke_chunk:15',
 'nke_chunk:16',
 'nke_chunk:17',
 'nke_chunk:18',
 'nke_chunk:19',
 'nke_chunk:20',
 'nke_chunk:21',
 'nke_chunk:22',
 'nke_chunk:23',
 'nke_chunk:24',
 'nke_chunk:25',
 'nke_chunk:26',
 'nke_chunk:27',
 'nke_chunk:28',
 'nke_chunk:29',
 'nke_chunk:30',
 'nke_chunk:31',
 'nke_chunk:32',
 'nke_chunk:33',
 'nke_chunk:34',
 'nke_chunk:35',
 'nke_chunk:36',
 'nke_chunk:37',
 'nke_chunk:38',
 'nke_chunk:39',
 'nke_chunk:40',
 'nke_chunk:41',
 'nke_chunk:42',
 'nke_chunk:43',
 'nke_chunk:44',
 'nke_chunk:45',
 'nke_chunk:46',
 'nke_chunk:47',
 'nke_chunk:48',
 'nke_chunk:49',
 'nke_chunk:50',
 'nke_chunk:51',
 'nke_chunk:52',
 'nke_chunk:53',
 'nke_chunk:54',
 'nke_chunk:55',
 'nke_chunk:56',
 'nke_chunk:57',
 'nke_chunk:58',
 'nke_c

Here’s what we are doing:

1. We loop through each chunk and its corresponding vector in embeddings.

2. For each chunk, we create a dictionary with:

* chunk_id: a unique ID based on its index in the list (this will be used as the Redis key)

* content: the actual text content of the chunk

* text_embedding: the vector for that chunk, converted to a binary buffer using array_to_buffer() (this is required for Redis hashes)

The result is a list of dictionaries — one per chunk — with each associated with a key that follows the pattern defined by the schema (e.g., chunk:80)1. 

## Verify the setup with vector search

####  KNN Search

Let's implement a KNN search. Write a KNN search to find the top 5 most relevant chunks from the Nike 10-K PDF based on the query "What were Nike's profit margins and company performance?".

A few things to note as you write your code:

1. Remember to correctly import the appropriate query class from redisvl.query.
2. The query object should:

* Have return fields for "chunk_id" and "content"
* A vector field name of "text_embedding"
* Return the score

In [11]:
from redisvl.query import VectorQuery

query = "What were Nike's profit margins and company performance"

query_embedding = hf.embed(query)

vector_query = VectorQuery(
    vector=query_embedding,
    vector_field_name="text_embedding",
    num_results=5,
    return_fields=["chunk_id", "content"],
    return_score=True
)

# execute the query with RedisVL
result=index.query(vector_query)

# view the results
pd.DataFrame(result)

,id,vector_distance,chunk_id,content
0,nke_chunk:80,0.302897632122,80,Table of Contents\nCONSOLIDATED OPERATING RESU...
1,nke_chunk:87,0.304227948189,87,Table of Contents\nOPERATING SEGMENTS\nAs disc...
2,nke_chunk:88,0.309843361378,88,"Greater China 2,283 2,365 -3 % 3,243 -27 %\nAs..."
3,nke_chunk:27,0.359879732132,27,"• In the past, certain retailers of our produc..."
4,nke_chunk:82,0.360612332821,82,Table of Contents\nFISCAL 2023 NIKE BRAND REVE...


## Build a RAG Pipeline

Notice that when we performed KNN search above to retrieve relevant chunks from Redis, the chunks themselves weren't very useful in generating a meaningful response to the query. The real power comes when we pass them as context to an LLM to answer the query.

In this section, we'll write the logic to accept a user question, search Redis for relevant context, and use that context to generate a response with a language model.

## Set up RedisVL AsyncSearchIndex

So far, we have used `SearchIndex` from RedisVL to create and query an index. When used, this class was performing operations in a synchronous (blocking) manner. That was perfectly fine for our exploratory code, but real-world applications — especially anything with a UI or concurrent workloads - benefit from non-blocking operations.

The good news is that RedisVL also offers an `AsyncSearchIndex` class. This class lets us query Redis using Python’s `async` and `await` syntax.

Below, we create the new index using the class, reusing the same schema from earlier. 

In [12]:
from redisvl.index import AsyncSearchIndex

async_index = AsyncSearchIndex.from_dict(schema, redis_url=REDIS_URL)

print("Async Index created successfully:", async_index)

Async Index created successfully: AsyncSearchIndex(name='doc_redisvl', prefixes=['nke_chunk'], storage_type='hash')


## Set up Ollama

In [13]:
CHAT_MODEL= "qwen2.5-coder:7b"

In [14]:
import ollama

def generate_rag_answer(question: str, retrieved_docs: list[str]) -> str:
    context = "\n\n".join(retrieved_docs)

    response = ollama.chat(
        model=CHAT_MODEL,
        messages=[
            {
                "role": "system",
                "content": """
                You are a helpful RAG assistant.
                Answer using only the supplied context.
                If the context does not contain the answer,
                clearly state that you don't know.
                Do not invent facts.
                """
            },
            {
                "role": "user",
                "content": f"""
                Retrieved context:
                {context}

                Question:
                {question}

                Answer:
                """
            }
        ],
        options={"temperature": 0.1}
    )

    return response["message"]["content"]


docs = [
    "Semantic caching stores responses to semantically similar queries.",
    "It can reduce repeated LLM calls and improve response latency."
]

answer = generate_rag_answer(
    "What is semantic caching, and what are its benefits?",
    docs
)

print(answer)


Semantic caching stores responses to semantically similar queries. It can reduce repeated LLM calls and improve response latency.


In [15]:
SYSTEM_PROMPT = """You are a helpful financial analyst assistant that has access
to public financial 10k documents in order to answer user's questions about company
performance, ethics, characteristics, and core information."""

## Query embedding

In [16]:
def embed_query(query: str):
    """Convert a user query into a dense vector representation."""
    return hf.embed(query)

## Retrieve relevant chunks from Redis

In [17]:
async def retrieve_context(async_index: AsyncSearchIndex, query_vector) -> str:
    """Fetch the relevant context from Redis using vector search"""
    
    results = await async_index.query(
        VectorQuery(
            vector=query_vector,
            vector_field_name="text_embedding",
            return_fields=["content"],
            num_results=3
        )
    )

    retrieved_context = "\n".join([result["content"] for result in results])
    return retrieved_context

## Constructing the prompt


Before we can call the LLM, we'll just need one more helper function to combine the user query and the retrieved context. We'll define a helper function called promptify() to do just that.

In [18]:
def promptify(query: str, context: str) -> str:
    return f'''Use the provided context below derived from public financial
    documents to answer the user's question. If you can't answer the user's
    question, based on the context; do not guess. If there is no context at all,
    respond with "I don't have enough context to answer this question.".

    User question:

    {query}

    Helpful context:

    {context}

    Answer:
    '''

## LLM Response

The final part of our RAG flow is to generate an answer using the LLM. In an earlier step, we configured our Gorq API credentials and imported the openai module. The module provides a class called AsyncClient to send the query and context to a model like gpt-4o-mini and return the response.

To make this clean and reusable, we’ll define a helper function called generate_llm_response(). 

In [19]:
import ollama

client = ollama.AsyncClient()

async def generate_llm_response(query: str, context: str) -> str:
    """Construct a prompt and generate a response using Ollama."""

    response = await client.chat(
        model="qwen2.5-coder:7b",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": promptify(query, context)}
        ],
        options={
            "temperature": 0.1,
            "seed": 42,
        },
    )

    # Return only the generated response
    return response["message"]["content"]


In [20]:
async def answer_question(index: AsyncSearchIndex, query: str):
    """End-to-end RAG: embeds query, retrieves context, generates LLM response"""

    # Step 1: Embed the query
    query_vector = embed_query(query)

    # Step 2: Retrieve matching context chunks from Redis
    context = await retrieve_context(index, query_vector)

    # Step 3: Generate response from OpenAI using system prompt + context
    return await generate_llm_response(query, context)

Question:

1. What is the trend in the company's revenue and profit over the past few years?
2. What are the company's primary revenue sources?
3. How much debt does the company have, and what are its capital expenditure plans?
4. What does the company say about its environmental, social, and governance (ESG) practices?

In [21]:
from chat_interface import start_chat

start_chat(answer_question, async_index)

## Adding RedisVL semantic caching

With a semantic cache, before we make a new LLM request, the system will embed the input and run a search in the cache. If there’s a “close” match (as defined by the cache settings), the system returns the cached answer and skips the LLM call entirely. That means zero LLM cost, near-instant response time, and more efficient operation.

In [22]:
from redisvl.extensions.cache.llm import SemanticCache

llmcache = SemanticCache(
    name="llmcache",                
    vectorizer=hf,                  
    redis_url=REDIS_URL,          
    ttl=300,                       
    distance_threshold=0.2,        
    overwrite=True      
)

print("Cache created successfully:", llmcache)

Cache created successfully: SemanticCache(name='llmcache', distance_threshold=0.2, ttl=300)


### Semantic cache parameter breakdown
We have included the following properties for the cache:

* name: An optional argument that sets the name of the semantic cache search index. Defaults to “llmcache”.
* vectorizer: An optional argument that sets the vectorizer for the cache.
* ttl: An optional argument that sets the "time to live" in seconds for a cached result. After 2 minutes, entries will expire.
* distance_threshold: The maximum distance for considering a cache hit. This defaults to 0.1.
* overwrite: Whether or not to force overwrite the schema for the semantic cache index. Defaults to false.

#### Examine the index with RedisVL CLI

In [23]:
!rvl index info -i llmcache



Index Information:
╭───────────────┬───────────────┬───────────────┬───────────────┬───────────────╮
│ Index Name    │ Storage Type  │ Prefixes      │ Index Options │ Indexing      │
├───────────────┼───────────────┼───────────────┼───────────────┼───────────────┤
| llmcache      | HASH          | ['llmcache']  | []            | 0             |
╰───────────────┴───────────────┴───────────────┴───────────────┴───────────────╯
Index Fields:
╭─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────╮
│ Name            │ Attribute       │ Type            │ Field Option    │ Option Value    │ Field Option    │ Option Value    │ Field Option    │ Option Value    │ Field Option    │ Option Value    │
├─────────────────┼─────────────────┼─────────────────┼─────────────────┼─────────────────┼─────────────────┼─────────────────┼─────────────────┼──────────

## Implementing the cache in the RAG app

There are a few different ways that the cache can be built into the existing app from the previous lab. We're going to keep it simple and create a new function called `answer_question_with_cache()` and use our previously defined `answer_question()` function inside of it.

In [24]:
import time

async def answer_question_with_cache(index, query_text):

    # Grab the start time for performance measurement
    start_time = time.time()

    # Embed the query text
    query_vector = embed_query(query_text)

    # Check cache
    result = llmcache.check(vector=query_vector)

    # If cache hit: set the response and return
    if result:
        elapsed = time.time() - start_time
        return {
            "text": result[0]['response'],
            "cached": True,
            "elapsed": elapsed
        }

    # If cache miss: go through full RAG pipeline
    response = await answer_question(index, query_text)
    llmcache.store(query_text, response, query_vector)

    # Calculate elapsed time and return the response
    elapsed = time.time() - start_time
    return {
        "text": response,
        "cached": False,
        "elapsed": elapsed
    }

Notice that we used two methods with our cache:

* .check(): This method checks the semantic cache for results similar to the specified prompt or vector. If a close enough match is found (based on the distance threshold), it returns the cached response; otherwise, it returns an empty list ([]).

* .store(): This method stores the specified key-value pair in the cache along with metadata. In our case, we are storing the query, its response, and the associated embedding vector.

We also changed up the implementation to visualize the speed at which our response will return. This will help you see how much faster the program runs with a cache. Feel free to check out how this works with the UI interface in the chat_ui.py file (around line 99) inside of the resources-rag directory.

### Test the cache out

With the cache implemented, we can now test the functionality. Below is a code block that will boot up the UI chat interface from the previous lab. Run the code block to create the chat interface and then try asking the two queries below to see the cache in action.

1. What was Nike's revenue last year compared to this year?
2. What was Nike's total revenue in the last year compared to now?

In [25]:
import sys
sys.path.append('resources-rag')
from chat_interface import start_chat

start_chat(answer_question_with_cache, async_index)

## Adding Memory

With caching implemented, we can tackle the issue of memory. Our chat still answers each question independently — with no awareness of what came before (other than if the query is "similar" to one in the cache).

To make our app more conversational and personalized, we'll want to preserve chat history. This will allow our chat app to understand follow-up questions or references to earlier answers.

In this section, we’ll implement storing the chat history using RedisVL's SemanticMessageHistory class.

## RedisVL SemanticMessageHistory

Let's import the class responsible for helping with session history: SemanticMessageHistory.

In [26]:
from redisvl.extensions.message_history import SemanticMessageHistory

#### A reusable assistant

At this point, we can start treating our RAG logic as a reusable assistant — so we'll wrap it in a class. This gives us a clean way to manage state (like session memory), reuse components (like the vectorizer and index), and organize our logic for answering questions more effectively.

Once the class is complete, we'll be able to call it like so:

    chat = ChatBot(async_index, vectorizer=hf, user="Tom")

We'll also build the ability to run two primary methods on the class instance:

1. .answer_question(): We'll use this to run the full RAG flow with caching and session memory. We'll upgrade the logic of our existing caching version of this function from earlier.
2. .clear_history(): We'll use this to reset the conversation memory for a clean slate

To start, let's create the class constructor. Using the structure above, it will need to take in an index, a vectorizer object, and a user name. We'll also define the session manager as a property of our class.

In [27]:
class ChatBot:
    def __init__(self, index, vectorizer, user: str):
        self.index = index
        self.vectorizer = vectorizer
        self.session_manager = SemanticMessageHistory(
            name=f"chat_session_{user}",
            redis_url=REDIS_URL, # redis_url was imported from the setup script
        )

The important argument to call out from above is name. This property defines the name of the Redis index where messages will be stored. In this example, we include the user's name in the index name (e.g., chat_session_tom) to give each user their own dedicated index.

An additional property to be aware of that we didn't assign above is the session_tag property. Setting this property can distinguish individual sessions. When it is not filled, it will default to a instance ULID, but it would be common to want to use the tag to describe what each different user session is about (e.g., tesla_stock_analysis, meta_ethics).

### ----- Attaching existing methods to the new class -----

Next, we'll attach the existing methods we built in the above. This includes slightly modified versions (to fit the class structure) of `embed_query()`, `retrieve_content()`, and `generate_llm_response()`. Visit the setup-lab-prod.py file to check them out.

In [28]:

def embed_query_class(self, query: str):
    """Convert a user query into a dense vector representation."""
    return self.vectorizer.embed(query)

#-------
async def retrieve_context_class(self, query_vector) -> str:
    """Fetch the relevant context from Redis using vector search"""
    results = await self.index.query(
        VectorQuery(
            vector=query_vector,
            vector_field_name="text_embedding",
            return_fields=["content"],
            num_results=3
        )
    )
    return "\n".join([result["content"] for result in results])

#------
async def generate_llm_response(query: str, context: str, session: list = None) -> str:
    """Construct and send a complete prompt to the LLM and return its response."""

    messages = (
        [{"role": "system", "content": SYSTEM_PROMPT}] +
        (session or []) +
        [{"role": "user", "content": promptify(query, context)}]
    )

    response = await client.chat(
        model=CHAT_MODEL,
        messages=messages,
        options= {"temperature":0.1, 
                "seed": 42,
        }
    )

    return response.message.content
    

In [29]:
ChatBot.embed_query = embed_query_class
ChatBot.retrieve_context = retrieve_context_class
ChatBot.generate_llm_response = staticmethod(generate_llm_response)

## Clear chat history

As mentioned earlier, one piece of functionality we'll also want as part of our class is the ability to clear the session history. This will enable us to start new sessions with a clean slate. This can be implemented by using the provided .clear() method from the SemanticMessageHistory class.

#### Clear history method

Create a new function in the code block below called clear_history. The method should call .clear() on the session manager object we created earlier. Then, on a separate line, attach the method to be part of the ChatBot class.

In [30]:
async def clear_history(self):
    """Clear session chat"""
    self.session_manager.clear()

ChatBot.clear_history = clear_history

### Query functionality

Lastly, but most importantly, we'll need to build the class method (defined as a function and then attached) for answering the query passed to the chat.

In [31]:
async def answer_question_with_cache_and_history(self, query: str):
    """Answer the user's question with session context and caching baked-in"""

    # Start timer to track performance
    start_time = time.time()

    # Initialize flags for cache and memory usage
    used_cache = False
    used_memory = False

    # 📌 Task A1: Embed the query using class helper method
    query_vector = self.embed_query(query)

    # 📌 Task A2: Check the semantic cache
    result = llmcache.check(vector=query_vector)

    # If cache hit, retrieve the answer and set the used_cache flag
    if result:

        answer = result[0]['response']
        used_cache = True

    else:
        # Retrieve current session messages
        session = self.session_manager.messages
        used_memory = bool(session)


        # 📌 Task A3: Retrieve context using class helper
        context = await self.retrieve_context(query_vector)
    
        # 📌 Task A4: Generate answer using LLM
        async def generate_llm_response(query: str, context: str, session: list) -> str:
            """Construct and send a complete prompt to the LLM and return its response."""

            messages = (
                [{"role": "system", "content": SYSTEM_PROMPT}] +
                session +
                [{"role": "user", "content": promptify(query, context)}]
            )

        answer = await self.generate_llm_response(query, context, session)
        
        # 📌 Task A5: Store response in semantic cache
        llmcache.store(query, answer, query_vector)
        
    # Update session memory and return the answer
    self.session_manager.add_messages([
        {"role": "user", "content": query},
        {"role": "assistant", "content": answer}
    ])

    # Return answer + metadata
    return {
        "text": answer,
        "cached": used_cache,
        "used_memory": used_memory,
        "elapsed": time.time() - start_time
    }

# Attaching the new method to the ChatBot class
ChatBot.answer_question_with_cache_and_history = answer_question_with_cache_and_history

## Test the entire RAG workflow

#### Want to test semantic caching again? 🔁
These two questions are phrased differently but ask for the same information. The second call should result in a cache hit, making the response nearly instant.

* What are Nike’s main operating segments?
* Can you describe Nike’s primary operating divisions and regions?

#### Want to test session memory? 🧠
Try any of the following sequences of related questions:

A.
1. What was Nike's marketing strategy?
2. What role do athlete partnerships play in that?

B.
1. What are Nike’s main revenue drivers according to its 2023 10-K?
2. Tell me more about the direct sales.

In [32]:
# Set up session
# Feel free to change the user name to your own
chat = ChatBot(async_index, vectorizer=hf, user="Tom")
await llmcache.aclear()
await chat.clear_history()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [33]:
from chat_interface import start_chat_advanced

start_chat_advanced(chat)